# Figure 1, panels b–d — regulators and gene programs during arterial endothelial differentiation

This notebook contains the source-verified computational producers for panels **b–d**. Panel a is omitted because it is an assembled schematic rather than a computational panel.

- **Panel b:** `Pos_vs_Neg_volcano_plotnine_MAGECK_FDR_pathway_colored_v8.py`.
- **Panel c:** recalculated directly from Chloe McCreery's validation-screen guide-count table on Sherlock. Guide-level EC percentages are reconstructed from the two CDH5-positive and CDH5-negative count pairs, and gene-level significance is assessed using two-sided Welch t-tests with BH correction across 297 genes.
- **Panel d:** Sherlock `programDE/notebooks/per_day_umaps/jesse_umap.ipynb`, using the integrated UMAP and cNMF H5MU object.

Machine-specific inputs are read from `config/paths.local.yaml`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import matplotlib.pyplot as plt

from src.cc_perturbseq.figure_notebook import (
    get_path, initialize_figure, load_figure_manifest, panel_directory, sha256_file, write_receipt,
)
from src.cc_perturbseq.figure1_panel_b import make_figure_01b
from src.cc_perturbseq.figure1_panel_c import PANEL_C_GROUPS, run_figure_01c
from src.cc_perturbseq.figure1_panel_d import make_figure_01d


def save_figure(fig, outdir: Path, stem: str, *, png_dpi: int = 600, pad_inches: float = 0.03):
    outputs = {}
    for suffix in ('pdf', 'svg', 'png'):
        path = outdir / f'{stem}.{suffix}'
        fig.savefig(
            path, dpi=png_dpi if suffix == 'png' else None,
            bbox_inches='tight', pad_inches=pad_inches, facecolor='white',
        )
        outputs[suffix] = str(path)
    return outputs

FIGURE = load_figure_manifest('Figure_01')
FIGURE['panels'] = [panel for panel in FIGURE['panels'] if panel['panel'] in {'b', 'c', 'd'}]
for panel in FIGURE['panels']:
    if panel['panel'] == 'c':
        panel['required_inputs'] = ['screen.validation_guide_counts']
FIGURE['scope'] = 'Panels b–d'
ctx = initialize_figure(FIGURE)
ctx['input_audit']


## Panel b — genome-wide CRISPRi screen

The x axis is the U-test log2 fold change. The y axis uses the directional MAGeCK FDR: `neg|fdr` for negative effects and `pos|fdr` for positive effects. A gene is a MAGeCK hit when either directional FDR is below 0.05.


In [ ]:
mageck_path = get_path(ctx['paths'], 'screen.genomewide_mageck_pos_vs_neg')
utest_path = get_path(ctx['paths'], 'screen.genomewide_utest_pos_vs_neg')
fig_b, source_b, labels_b = make_figure_01b(mageck_path, utest_path)
out_b = panel_directory(ctx, 'b')
outputs_b = save_figure(fig_b, out_b, 'Figure_01b_genomewide_screen_volcano')
source_b.to_csv(out_b / 'Figure_01b_source_data.csv', index=False)
labels_b.to_csv(out_b / 'Figure_01b_label_positions.csv', index=False)
write_receipt(
    ctx, 'b', status='SOURCE_VERIFIED_IMPLEMENTATION',
    inputs={
        'mageck': str(mageck_path), 'mageck_sha256': sha256_file(mageck_path),
        'utest': str(utest_path), 'utest_sha256': sha256_file(utest_path),
    },
    outputs={**outputs_b, 'source_data': str(out_b / 'Figure_01b_source_data.csv')},
    parameters={
        'source_script': 'Pos_vs_Neg_volcano_plotnine_MAGECK_FDR_pathway_colored_v8.py',
        'x_axis': 'U-test log2FC', 'y_axis': 'directional MAGeCK FDR',
        'fdr_cutoff': 0.05, 'vertical_reference': 0.0,
    },
)
plt.show()


## Panel c — validation-screen guide-level Day 3 EC fractions

The panel is recalculated from Chloe's guide-count table:

`/oak/stanford/groups/engreitz/Users/chloemc/GenomeWideValidation/MageckAnalysis/mageck_sgRNA_readcount.txt`

For each biological replicate, a 0.5-count pseudocount is added and the positive/negative count ratio is converted to log2 odds. The replicate-specific odds are centered so that the arithmetic mean reconstructed EC percentage across the 120 control guides is 70%. The two replicate-specific percentages are then averaged for each guide. Each target gene is compared with the 120 controls using a **two-sided Welch t-test**, followed by BH correction across all 297 target genes.


In [ ]:
validation_counts_path = get_path(ctx['paths'], 'screen.validation_guide_counts')
out_c = panel_directory(ctx, 'c')
(
    fig_c,
    stats_c,
    selected_stats_c,
    guide_table_c,
    outputs_c,
    validation_counts_hash,
    calibration_offsets_c,
) = run_figure_01c(
    validation_counts_path,
    out_c,
    control_baseline_pct=70.0,
    pseudocount=0.5,
)
write_receipt(
    ctx, 'c', status='SOURCE_VERIFIED_RECALCULATION',
    inputs={
        'validation_guide_counts': str(validation_counts_path),
        'sha256': validation_counts_hash,
    },
    outputs=outputs_c,
    parameters={
        'source': 'Chloe McCreery validation-screen MAGeCK guide-count table',
        'positive_columns': ['D3PR1', 'D3PR2'],
        'negative_columns': ['D3NR1', 'D3NR2'],
        'count_pseudocount': 0.5,
        'control_baseline_pct': 70.0,
        'calibration': 'replicate-specific offset chosen so mean of 120 reconstructed control-guide percentages equals 70%',
        'replicate_combination': 'arithmetic mean of reconstructed replicate-specific percentages per guide',
        'test': 'two-sided Welch t-test, target-guide values versus 120 control-guide values',
        'multiple_testing': 'Benjamini-Hochberg across 297 target genes',
        'weighting': 'one unweighted observation per guide',
        'calibration_offsets': calibration_offsets_c,
        'plotted_groups': PANEL_C_GROUPS,
    },
)
plt.show()
selected_stats_c[[
    'gene', 'n_targeting_guides', 'mean_target_EC_pct',
    'welch_pvalue_two_sided', 'FDR_BH', 'significance',
]]


## Panel d — integrated UMAP colored by dominant selected cNMF program

This is a lean port of `per_day_umaps/jesse_umap.ipynb`. It transfers `X_umap` from the integrated AnnData object after exact cell-ID verification, then assigns each cell to the highest-scoring program listed in `program_order.txt`. Cells with a maximum selected-program score of 0.02 or less are gray.


In [ ]:
h5mu_path = get_path(ctx['paths'], 'ccperturbseq.cnmf_h5mu')
adata_path = get_path(ctx['paths'], 'ccperturbseq.integrated_anndata')
metadata_path = get_path(ctx['paths'], 'ccperturbseq.figure1d_program_metadata')
order_path = get_path(ctx['paths'], 'ccperturbseq.figure1d_program_order')
fig_d, source_d, programs_d, threshold_d = make_figure_01d(
    h5mu_path, adata_path, metadata_path, order_path,
)
out_d = panel_directory(ctx, 'd')
outputs_d = save_figure(
    fig_d, out_d, 'Figure_01d_integrated_program_atlas', png_dpi=300, pad_inches=0.02,
)
source_d.to_csv(
    out_d / 'Figure_01d_cell_program_assignment.csv.gz',
    index=False, compression='gzip',
)
write_receipt(
    ctx, 'd', status='SOURCE_VERIFIED_IMPLEMENTATION',
    inputs={
        'h5mu': str(h5mu_path), 'integrated_anndata': str(adata_path),
        'program_metadata': str(metadata_path), 'program_order': str(order_path),
    },
    outputs={
        **outputs_d,
        'cell_program_assignment': str(out_d / 'Figure_01d_cell_program_assignment.csv.gz'),
    },
    parameters={
        'source_notebook': 'programDE/notebooks/per_day_umaps/jesse_umap.ipynb',
        'programs': programs_d, 'dominant_program_threshold': threshold_d,
        'umap_source': 'integrated AnnData X_umap',
        'program_score_source': 'cNMF modality in cNMF_50_0_2.h5mu',
    },
)
plt.show()
source_d[['dominant_selected_program', 'above_0.02_threshold']].value_counts().sort_index()
